# Fixture 생산계획 대시보드
**5월 생산계획.xlsx** — Fixture 총 발주수량 시트 기준

In [3]:
import pandas as pd
import numpy as np
import openpyxl
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# JupyterLab 인라인 렌더링

EXCEL_FILE = "5월 생산계획.xlsx"
SHEET_NAME = "Fixture 총 발주수량"
from IPython.display import IFrame
import os
os.makedirs("iframe_figures", exist_ok=True)

def show_fig(fig, fname=None, height=520):
    html = fig.to_html(full_html=False, include_plotlyjs="cdn")
    display(HTML(f'<div style="height:{height}px;width:100%">{html}</div>'),
            metadata={"isolated": True})


In [4]:
# ─────────────────────────────────────────
# 데이터 로드
# ─────────────────────────────────────────
wb = openpyxl.load_workbook(EXCEL_FILE, data_only=True)
ws = wb[SHEET_NAME]

CE_ORDER_COLS  = {'C': '아자르바이젠', 'D': '베트남', 'E': '이라크PO30',
                  'F': '리투아니아',   'G': '이라크PO31', 'H': '두바이', 'I': '몰도바'}
FDA_ORDER_COLS = {'K': '이란'}

WIP_COLS = {
    'Q': '멸균완료\n포장대기',
    'S': '멸균중',
    'U': '멸균대기',
    'W': '클린룸',
    'Y': '2차검사',
    'AA': '샌딩&에칭',
    'AC': '샌딩대기',
    'AD': '1차검사',
    'AF': 'CNC가공',
}

def col(letter):
    return openpyxl.utils.column_index_from_string(letter)

def cell_val(row, letter):
    v = ws.cell(row=row, column=col(letter)).value
    return v if v is not None else 0

records = []
for r in range(4, 45):
    code = ws.cell(row=r, column=col('B')).value
    if not code:
        continue

    rec = {'품목코드': code}

    # CE 주문 (시장별)
    for c_letter, market in CE_ORDER_COLS.items():
        rec[f'CE_{market}'] = cell_val(r, c_letter) or 0
    rec['CE_주문합계'] = cell_val(r, 'J') or 0

    # FDA 주문
    for c_letter, market in FDA_ORDER_COLS.items():
        rec[f'FDA_{market}'] = cell_val(r, c_letter) or 0
    rec['FDA_주문합계'] = cell_val(r, 'L') or 0

    # 재고
    rec['포장재고_CE']  = cell_val(r, 'M') or 0
    rec['포장재고_FDA'] = cell_val(r, 'N') or 0

    # 출하가능
    rec['출하가능_CE']  = cell_val(r, 'O') or 0
    rec['출하가능_FDA'] = cell_val(r, 'P') or 0
    rec['총_출하가능']  = cell_val(r, 'AH') or 0

    # 공정별 재공
    for c_letter, proc_name in WIP_COLS.items():
        rec[f'WIP_{proc_name}'] = cell_val(r, c_letter) or 0

    records.append(rec)

df = pd.DataFrame(records)

# 파생 컬럼
df['총_주문'] = df['CE_주문합계'] + df['FDA_주문합계']
df['총_포장재고'] = df['포장재고_CE'] + df['포장재고_FDA']

wip_cols_display = [f'WIP_{v}' for v in WIP_COLS.values()]
df['총_재공'] = df[wip_cols_display].sum(axis=1)

# 부족/여유 상태 분류
def shortage_level(row):
    if row['총_출하가능'] >= 0:
        return '충족'
    elif row['총_출하가능'] >= -row['총_주문'] * 0.3:
        return '소부족'
    else:
        return '대부족'

df['부족상태'] = df.apply(shortage_level, axis=1)

print(f'✅ 로드 완료: {len(df)}개 품목')
print(f'   - 충족: {(df["부족상태"]=="충족").sum()}개')
print(f'   - 소부족 (30% 미만): {(df["부족상태"]=="소부족").sum()}개')
print(f'   - 대부족 (30% 이상): {(df["부족상태"]=="대부족").sum()}개')

✅ 로드 완료: 41개 품목
   - 충족: 10개
   - 소부족 (30% 미만): 11개
   - 대부족 (30% 이상): 20개


## 1. 전체 현황 요약

In [5]:
# KPI 카드
total_ce_order  = df['CE_주문합계'].sum()
total_fda_order = df['FDA_주문합계'].sum()
total_ce_stock  = df['포장재고_CE'].sum()
total_fda_stock = df['포장재고_FDA'].sum()
total_wip       = df['총_재공'].sum()
n_shortage      = (df['부족상태'] != '충족').sum()

kpi_html = f"""
<style>
  .kpi-wrap {{ display:flex; gap:14px; flex-wrap:wrap; font-family:sans-serif; }}
  .kpi {{ background:#1e2130; border-radius:10px; padding:18px 24px; min-width:160px; text-align:center; }}
  .kpi .label {{ font-size:12px; color:#8892b0; margin-bottom:6px; }}
  .kpi .value {{ font-size:28px; font-weight:700; }}
  .kpi .sub   {{ font-size:11px; color:#8892b0; margin-top:4px; }}
  .blue   {{ color:#64b5f6; }} .green {{ color:#69db7c; }}
  .orange {{ color:#ffa94d; }} .red   {{ color:#ff6b6b; }}
  .purple {{ color:#da77f2; }} .teal  {{ color:#38d9a9; }}
</style>
<div class="kpi-wrap">
  <div class="kpi"><div class="label">CE 총 주문</div><div class="value blue">{total_ce_order:,}</div><div class="sub">PCS</div></div>
  <div class="kpi"><div class="label">FDA 총 주문</div><div class="value purple">{total_fda_order:,}</div><div class="sub">PCS</div></div>
  <div class="kpi"><div class="label">CE 포장재고</div><div class="value green">{total_ce_stock:,}</div><div class="sub">PCS</div></div>
  <div class="kpi"><div class="label">FDA 포장재고</div><div class="value teal">{total_fda_stock:,}</div><div class="sub">PCS</div></div>
  <div class="kpi"><div class="label">전체 재공</div><div class="value orange">{total_wip:,}</div><div class="sub">PCS</div></div>
  <div class="kpi"><div class="label">부족 품목수</div><div class="value red">{n_shortage}</div><div class="sub">/ {len(df)} 품목</div></div>
</div>
"""
display(HTML(kpi_html))

## 2. 품목별 주문 vs 재고 현황

In [6]:
color_map = {'충족': '#69db7c', '소부족': '#ffa94d', '대부족': '#ff6b6b'}

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('CE향 — 주문 vs 포장재고', 'FDA향 — 주문 vs 포장재고'),
    vertical_spacing=0.12
)

bar_color = [color_map[s] for s in df['부족상태']]

# CE
fig.add_trace(go.Bar(
    name='CE 주문', x=df['품목코드'], y=df['CE_주문합계'],
    marker_color='#64b5f6', opacity=0.85
), row=1, col=1)
fig.add_trace(go.Bar(
    name='CE 포장재고', x=df['품목코드'], y=df['포장재고_CE'],
    marker_color='#69db7c', opacity=0.85
), row=1, col=1)

# FDA
fig.add_trace(go.Bar(
    name='FDA 주문', x=df['품목코드'], y=df['FDA_주문합계'],
    marker_color='#da77f2', opacity=0.85
), row=2, col=1)
fig.add_trace(go.Bar(
    name='FDA 포장재고', x=df['품목코드'], y=df['포장재고_FDA'],
    marker_color='#38d9a9', opacity=0.85
), row=2, col=1)

fig.update_layout(
    height=700, barmode='group',
    template='plotly_dark',
    legend=dict(orientation='h', y=1.04),
    margin=dict(l=40, r=20, t=80, b=120)
)
fig.update_xaxes(tickangle=45, tickfont=dict(size=9))
show_fig(fig, "fig01.html")

## 3. 부족량 분석 (출하가능 = 재공 포함 후 잔여)

In [7]:
df_sorted = df.sort_values('총_출하가능')

bar_colors = [color_map[s] for s in df_sorted['부족상태']]

fig = go.Figure(go.Bar(
    x=df_sorted['품목코드'],
    y=df_sorted['총_출하가능'],
    marker_color=bar_colors,
    text=df_sorted['총_출하가능'].apply(lambda v: f'{v:+,}'),
    textposition='outside',
    textfont=dict(size=9),
))
fig.add_hline(y=0, line_color='white', line_width=1.5)
fig.update_layout(
    title='품목별 총 출하가능 수량 (음수 = 부족)',
    template='plotly_dark',
    height=480,
    margin=dict(l=40, r=20, t=60, b=130),
    xaxis_tickangle=45,
    xaxis_tickfont=dict(size=9),
    yaxis_title='수량 (PCS)'
)
show_fig(fig, "fig02.html")

## 4. 공정별 재공 현황

In [8]:
proc_names  = list(WIP_COLS.values())
proc_totals = [df[f'WIP_{p}'].sum() for p in proc_names]

palette = px.colors.qualitative.Pastel

fig = go.Figure(go.Bar(
    x=proc_names,
    y=proc_totals,
    marker_color=palette[:len(proc_names)],
    text=[f'{v:,}' for v in proc_totals],
    textposition='outside',
))
fig.update_layout(
    title='공정별 재공 합계',
    template='plotly_dark',
    height=420,
    yaxis_title='수량 (PCS)',
    margin=dict(l=40, r=20, t=60, b=60)
)
show_fig(fig, "fig03.html")

# 도넛 차트 (비율)
non_zero = [(n, v) for n, v in zip(proc_names, proc_totals) if v > 0]
if non_zero:
    fig2 = go.Figure(go.Pie(
        labels=[n for n, _ in non_zero],
        values=[v for _, v in non_zero],
        hole=0.45,
        textinfo='label+percent',
    ))
    fig2.update_layout(
        title='공정별 재공 비율',
        template='plotly_dark',
        height=420,
        margin=dict(l=20, r=20, t=60, b=20)
    )
    show_fig(fig2, "fig04.html")

## 5. 품목별 공정 재공 히트맵

In [9]:
heatmap_data = df[['품목코드'] + wip_cols_display].set_index('품목코드')
heatmap_data.columns = proc_names
# 재공이 하나라도 있는 품목만
heatmap_data = heatmap_data[heatmap_data.sum(axis=1) > 0]

if not heatmap_data.empty:
    fig = go.Figure(go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns.tolist(),
        y=heatmap_data.index.tolist(),
        colorscale='YlOrRd',
        text=heatmap_data.values,
        texttemplate='%{text:,}',
        textfont=dict(size=9),
        colorbar=dict(title='수량'),
    ))
    fig.update_layout(
        title='품목 × 공정 재공 히트맵 (재공 있는 품목만)',
        template='plotly_dark',
        height=max(350, len(heatmap_data) * 22 + 120),
        margin=dict(l=140, r=20, t=60, b=60),
        xaxis_tickangle=30,
    )
    show_fig(fig, "fig05.html")
else:
    print('재공 데이터 없음')

## 6. CE 시장별 주문 구성

In [10]:
market_cols = [f'CE_{m}' for m in CE_ORDER_COLS.values()]
market_totals = {m: df[f'CE_{m}'].sum() for m in CE_ORDER_COLS.values()}
market_totals = {k: v for k, v in market_totals.items() if v > 0}

fig = go.Figure(go.Pie(
    labels=list(market_totals.keys()),
    values=list(market_totals.values()),
    hole=0.4,
    textinfo='label+value+percent',
    textfont=dict(size=11),
))
fig.update_layout(
    title='CE 시장별 주문 구성',
    template='plotly_dark',
    height=420,
    margin=dict(l=20, r=20, t=60, b=20)
)
show_fig(fig, "fig06.html")

## 7. 긴급 조치 필요 품목 (대부족)

In [11]:
df_urgent = df[df['부족상태'] == '대부족'].copy()
df_urgent = df_urgent.sort_values('총_출하가능')[[
    '품목코드', 'CE_주문합계', 'FDA_주문합계', '총_주문',
    '포장재고_CE', '포장재고_FDA', '총_포장재고',
    '총_재공', '출하가능_CE', '출하가능_FDA', '총_출하가능'
]]

def highlight_negative(val):
    if isinstance(val, (int, float)) and val < 0:
        return 'color: #ff6b6b; font-weight: bold'
    return ''

styled = (df_urgent.style
          .map(highlight_negative)
          .format({
              col: '{:,}' for col in df_urgent.select_dtypes('number').columns
          })
          .set_caption(f'⚠️ 긴급 부족 품목 ({len(df_urgent)}개)')
          .set_table_styles([{
              'selector': 'caption',
              'props': [('font-size', '15px'), ('font-weight', 'bold'), ('color', '#ff6b6b')]
          }])
         )
display(styled)

if df_urgent.empty:
    print('✅ 대부족 품목 없음')

,품목코드,CE_주문합계,FDA_주문합계,총_주문,포장재고_CE,포장재고_FDA,총_포장재고,총_재공,출하가능_CE,출하가능_FDA,총_출하가능
14,C1R4010C,"4,594","20,000","24,594","1,268",685,"1,953","12,860","-3,326","-19,315","-10,131"
24,C1R4510C,"1,475","14,000","15,475",0,55,55,"7,426","-1,475","-13,945","-8,234"
15,C1R4011C,"2,080","10,000","12,080",308,34,342,"5,592","-1,772","-9,966","-6,338"
25,C1R4511C,420,"5,000","5,420",324,57,381,0,-96,"-4,943","-5,169"
13,C1R4008C,"2,100","5,000","7,100",378,"1,515","1,893","1,524","-1,722","-3,485","-3,873"
23,C1R4508C,930,"3,000","3,930",237,447,684,0,-693,"-2,553","-3,390"
17,C1R4007SC,0,"3,000","3,000",0,63,63,0,0,"-2,937","-3,137"
6,C1M3511C,630,"2,000","2,630",178,47,225,0,-452,"-1,953","-2,549"
16,C1R4013C,315,"2,000","2,315",334,560,894,0,19,"-1,440","-1,480"
4,C1M3508C,"1,469","1,000","2,469",403,125,528,665,"-1,066",-875,"-1,401"


## 8. 전체 품목 상세 테이블

In [12]:
# 필터 위젯
status_filter = widgets.ToggleButtons(
    options=['전체', '대부족', '소부족', '충족'],
    description='상태 필터:',
    button_style='',
)

search_box = widgets.Text(
    placeholder='품목코드 검색...',
    description='품목코드:',
    layout=widgets.Layout(width='300px')
)

out = widgets.Output()

display_cols = [
    '품목코드', 'CE_주문합계', 'FDA_주문합계', '총_주문',
    '포장재고_CE', '포장재고_FDA', '총_재공',
    '출하가능_CE', '출하가능_FDA', '총_출하가능', '부족상태'
]

def refresh(*args):
    out.clear_output(wait=True)
    filtered = df.copy()
    if status_filter.value != '전체':
        filtered = filtered[filtered['부족상태'] == status_filter.value]
    if search_box.value.strip():
        filtered = filtered[filtered['품목코드'].str.contains(
            search_box.value.strip(), case=False, na=False
        )]
    filtered = filtered[display_cols].sort_values('총_출하가능')

    color_status = {'충족': '#69db7c', '소부족': '#ffa94d', '대부족': '#ff6b6b'}

    def color_row(row):
        styles = [''] * len(row)
        for i, col_name in enumerate(row.index):
            if col_name == '부족상태':
                styles[i] = f'color: {color_status.get(row[col_name], "")}; font-weight:bold'
            elif isinstance(row[col_name], (int, float)) and row[col_name] < 0:
                styles[i] = 'color: #ff6b6b'
        return styles

    with out:
        if filtered.empty:
            print('조건에 맞는 품목이 없습니다.')
        else:
            styled_df = (filtered.style
                         .apply(color_row, axis=1)
                         .format({
                             c: '{:,}' for c in filtered.select_dtypes('number').columns
                         }))
            display(styled_df)
            print(f'총 {len(filtered)}개 품목')

status_filter.observe(refresh, names='value')
search_box.observe(refresh, names='value')

display(widgets.HBox([status_filter, search_box]))
display(out)
refresh()

Output()

## 9. 품목 상세 조회

In [13]:
item_dropdown = widgets.Dropdown(
    options=df['품목코드'].tolist(),
    description='품목 선택:',
    layout=widgets.Layout(width='350px')
)
detail_out = widgets.Output()

def show_detail(change):
    detail_out.clear_output(wait=True)
    code = item_dropdown.value
    row = df[df['품목코드'] == code].iloc[0]

    with detail_out:
        # Waterfall: 재고 흐름
        wip_vals = [row[f'WIP_{p}'] for p in proc_names if row[f'WIP_{p}'] > 0]
        wip_labels = [p for p in proc_names if row[f'WIP_{p}'] > 0]

        fig = go.Figure()

        # 주문 vs 재고+재공 막대
        fig.add_trace(go.Bar(
            name='CE 주문', x=['CE 주문'], y=[row['CE_주문합계']],
            marker_color='#64b5f6'
        ))
        fig.add_trace(go.Bar(
            name='FDA 주문', x=['FDA 주문'], y=[row['FDA_주문합계']],
            marker_color='#da77f2'
        ))
        fig.add_trace(go.Bar(
            name='CE 포장재고', x=['CE 포장재고'], y=[row['포장재고_CE']],
            marker_color='#69db7c'
        ))
        fig.add_trace(go.Bar(
            name='FDA 포장재고', x=['FDA 포장재고'], y=[row['포장재고_FDA']],
            marker_color='#38d9a9'
        ))
        for label, val in zip(wip_labels, wip_vals):
            fig.add_trace(go.Bar(
                name=label, x=[f'재공\n{label}'], y=[val],
                marker_color='#ffa94d'
            ))

        status_color = color_map[row['부족상태']]
        fig.update_layout(
            title=dict(
                text=f'{code} — 총 출하가능: <b><span style="color:{status_color}">{row["총_출하가능"]:+,}</span></b> PCS',
                font=dict(size=15)
            ),
            template='plotly_dark',
            height=380,
            barmode='group',
            showlegend=False,
            margin=dict(l=40, r=20, t=70, b=60)
        )
        show_fig(fig, "fig07.html")

        # CE 시장별 주문 상세
        ce_detail = {m: row[f'CE_{m}'] for m in CE_ORDER_COLS.values() if row[f'CE_{m}'] > 0}
        if ce_detail:
            display(HTML('<b>CE 시장별 주문:</b> ' + 
                         ' | '.join(f'{k}: <b>{v:,}</b>' for k, v in ce_detail.items())))

item_dropdown.observe(show_detail, names='value')
display(item_dropdown, detail_out)
show_detail(None)

Dropdown(description='품목 선택:', layout=Layout(width='350px'), options=('C1M3008C', 'C1M3010C', 'C1M3011C', 'C1M…

Output()

## 10. 우선순위 기반 재고 할당

아래 딕셔너리에서 각 주문의 우선순위를 직접 수정하세요.

In [14]:
# ─────────────────────────────────────────────────
# ★ 이 셀에서 우선순위를 직접 수정하세요
#   A = 긴급 (가장 먼저 배분)
#   B = 보통
#   C = 여유 (마지막 배분)
# ─────────────────────────────────────────────────

CE_PRIORITY = {
    'CE_아자르바이젠': 'D',
    'CE_베트남':       'A',
    'CE_이라크PO30':   'B',
    'CE_리투아니아':   'A',
    'CE_이라크PO31':   'C',
    'CE_두바이':       'C',
    'CE_몰도바':       'C',
}

# FDA 이란 주문 분할 (ratio 합계 비율로 수량 배분)
FDA_SPLITS = [
    {'name': 'FDA 이란 ①', 'ratio': 40000, 'priority': 'B'},
    {'name': 'FDA 이란 ②', 'ratio': 40000, 'priority': 'C'},
    {'name': 'FDA 이란 ③', 'ratio': 55000, 'priority': 'D'},
]

PRIORITY_ORDER = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
PRIORITY_LABEL = {'A': 'A 긴급', 'B': 'B 보통', 'C': 'C 여유', 'D': 'D 여유'}
FDA_TOTAL_RATIO = sum(s['ratio'] for s in FDA_SPLITS)

print("=== 우선순위 설정 ===")
print("[ CE 주문 ]")
for k, v in CE_PRIORITY.items():
    print(f"  {k:20s}: {PRIORITY_LABEL[v]}")
print("\n[ FDA 이란 분할 ]")
for s in FDA_SPLITS:
    pct = s['ratio'] / FDA_TOTAL_RATIO * 100
    print(f"  {s['name']}: {s['ratio']:,}개 ({pct:.1f}%) → {PRIORITY_LABEL[s['priority']]}")


=== 우선순위 설정 ===
[ CE 주문 ]
  CE_아자르바이젠           : D 여유
  CE_베트남              : A 긴급
  CE_이라크PO30          : B 보통
  CE_리투아니아            : A 긴급
  CE_이라크PO31          : C 여유
  CE_두바이              : C 여유
  CE_몰도바              : C 여유

[ FDA 이란 분할 ]
  FDA 이란 ①: 40,000개 (29.6%) → B 보통
  FDA 이란 ②: 40,000개 (29.6%) → C 여유
  FDA 이란 ③: 55,000개 (40.7%) → D 여유


In [15]:
# ─────────────────────────────────────────────────
# 할당 로직
# 배분 우선순위: A → B → C → D
# 같은 우선순위 내: CE 재고 → FDA 재고 → 공정 재공(WIP, 공용)
# ─────────────────────────────────────────────────

def allocate_item(row):
    ce_stock  = row['포장재고_CE']   # CE 전용
    fda_stock = row['포장재고_FDA']  # FDA 전용
    wip       = row['총_재공']       # CE/FDA 공용

    # CE 주문 목록
    ce_orders = []
    for col_name, priority in CE_PRIORITY.items():
        qty = row.get(col_name, 0)
        if qty > 0:
            ce_orders.append({'name': col_name, 'priority': priority,
                               'type': 'CE', 'ordered': qty})

    # FDA 주문 비례 분할
    fda_total = row.get('FDA_이란', 0)
    fda_orders = []
    remaining_fda = fda_total
    for i, split in enumerate(FDA_SPLITS):
        if i < len(FDA_SPLITS) - 1:
            qty = round(fda_total * split['ratio'] / FDA_TOTAL_RATIO)
        else:
            qty = remaining_fda   # 마지막은 잔량 처리(반올림 오차 방지)
        remaining_fda -= qty
        if qty > 0:
            fda_orders.append({'name': split['name'], 'priority': split['priority'],
                                'type': 'FDA', 'ordered': qty})

    # 우선순위 정렬 (A→B→C→D, 동순위면 CE 먼저)
    all_orders = sorted(
        ce_orders + fda_orders,
        key=lambda x: (PRIORITY_ORDER[x['priority']], 0 if x['type'] == 'CE' else 1)
    )

    ce_avail  = ce_stock
    fda_avail = fda_stock
    wip_avail = wip

    results = []
    for order in all_orders:
        needed    = order['ordered']
        allocated = 0

        if order['type'] == 'CE':
            use = min(needed, ce_avail); ce_avail -= use; allocated += use; needed -= use
            use = min(needed, wip_avail); wip_avail -= use; allocated += use; needed -= use
        else:
            use = min(needed, fda_avail); fda_avail -= use; allocated += use; needed -= use
            use = min(needed, wip_avail); wip_avail -= use; allocated += use; needed -= use

        shortage = needed
        results.append({
            **order,
            'allocated': allocated,
            'shortage': shortage,
            'fulfill_pct': round(allocated / order['ordered'] * 100, 1) if order['ordered'] else 100.0,
        })
    return results

# 전체 품목 적용
alloc_rows = []
for _, row in df.iterrows():
    for r in allocate_item(row):
        alloc_rows.append({'품목코드': row['품목코드'], **r})

df_alloc = pd.DataFrame(alloc_rows)

# 우선순위 라벨 추가
df_alloc['우선순위'] = df_alloc['priority'].map(PRIORITY_LABEL)

total_ordered  = df_alloc['ordered'].sum()
total_alloc    = df_alloc['allocated'].sum()
total_shortage = df_alloc['shortage'].sum()
print(f"할당 완료: {len(df_alloc)}건")
print(f"  총 주문: {total_ordered:,} | 총 할당: {total_alloc:,} | 총 부족: {total_shortage:,}")
print(f"  전체 충족률: {total_alloc/total_ordered*100:.1f}%")


할당 완료: 165건
  총 주문: 145,358 | 총 할당: 101,422 | 총 부족: 43,936
  전체 충족률: 69.8%


### 10-1. 우선순위별 충족 현황

In [16]:
summary = (df_alloc.groupby('priority')
           .agg(총주문=('ordered','sum'), 총할당=('allocated','sum'), 총부족=('shortage','sum'))
           .reset_index())
summary['충족률(%)'] = (summary['총할당'] / summary['총주문'] * 100).round(1)
summary['우선순위']  = summary['priority'].map(PRIORITY_LABEL)
summary = summary.sort_values('priority')

# 스택 막대 차트
fig = go.Figure()
fig.add_trace(go.Bar(
    name='할당 완료', x=summary['우선순위'], y=summary['총할당'],
    marker_color='#69db7c',
    text=summary['총할당'].apply(lambda v: f'{v:,}'),
    textposition='inside', textfont=dict(size=11)
))
fig.add_trace(go.Bar(
    name='부족', x=summary['우선순위'], y=summary['총부족'],
    marker_color='#ff6b6b',
    text=summary['총부족'].apply(lambda v: f'{v:,}' if v > 0 else ''),
    textposition='inside', textfont=dict(size=11)
))
for _, row in summary.iterrows():
    fig.add_annotation(
        x=row['우선순위'], y=row['총주문'] + 200,
        text=f"{row['충족률(%)']:.1f}%",
        showarrow=False, font=dict(size=13, color='white')
    )
fig.update_layout(
    barmode='stack', title='우선순위별 주문 충족 현황 (전 품목 합산)',
    template='plotly_dark', height=420,
    yaxis_title='수량 (PCS)',
    margin=dict(l=40, r=20, t=60, b=40)
)
show_fig(fig, "fig08.html")

display(summary[['우선순위','총주문','총할당','총부족','충족률(%)']].style
        .format({'총주문':'{:,}','총할당':'{:,}','총부족':'{:,}','충족률(%)':'{:.1f}%'})
        .background_gradient(subset=['충족률(%)'], cmap='RdYlGn', vmin=0, vmax=100))


,우선순위,총주문,총할당,총부족,충족률(%)
0,A 긴급,"3,006","2,943",63,97.9%
1,B 보통,"45,879","42,324","3,555",92.3%
2,C 여유,"41,402","31,564","9,838",76.2%
3,D 여유,"55,071","24,591","30,480",44.7%


### 10-2. 주문별 충족 현황 (전 품목 합산)

In [17]:
order_summary = (df_alloc.groupby(['우선순위','name','type'])
                 .agg(총주문=('ordered','sum'), 총할당=('allocated','sum'), 총부족=('shortage','sum'))
                 .reset_index())
order_summary['충족률(%)'] = (order_summary['총할당'] / order_summary['총주문'] * 100).round(1)
order_summary = order_summary.sort_values(['우선순위','type'])

fig = go.Figure()
colors_alloc = {'CE': '#64b5f6', 'FDA': '#da77f2'}
colors_short = {'CE': '#ff6b6b', 'FDA': '#ff9e7a'}

for order_type in ['CE', 'FDA']:
    sub = order_summary[order_summary['type'] == order_type]
    fig.add_trace(go.Bar(
        name=f'{order_type} 할당', x=sub['name'], y=sub['총할당'],
        marker_color=colors_alloc[order_type],
        text=sub['충족률(%)'].apply(lambda v: f'{v:.0f}%'),
        textposition='outside', textfont=dict(size=9)
    ))
    fig.add_trace(go.Bar(
        name=f'{order_type} 부족', x=sub['name'], y=sub['총부족'],
        marker_color=colors_short[order_type], opacity=0.7
    ))

fig.update_layout(
    barmode='stack', title='주문별 할당/부족 현황 (전 품목 합산)',
    template='plotly_dark', height=460,
    xaxis_tickangle=30, yaxis_title='수량 (PCS)',
    margin=dict(l=40, r=20, t=60, b=100)
)
show_fig(fig, "fig09.html")


### 10-3. 품목 × 주문 할당 상세

In [18]:
# 인터랙티브: 품목 선택 → 주문별 할당 상세
alloc_dropdown = widgets.Dropdown(
    options=df['품목코드'].tolist(),
    description='품목 선택:',
    layout=widgets.Layout(width='350px')
)
alloc_out = widgets.Output()

def show_alloc(change):
    alloc_out.clear_output(wait=True)
    code = alloc_dropdown.value
    sub = df_alloc[df_alloc['품목코드'] == code].copy()
    row = df[df['품목코드'] == code].iloc[0]

    with alloc_out:
        fig = go.Figure()
        clr = {'A 긴급': '#ff6b6b', 'B 보통': '#ffa94d', 'C 여유': '#69db7c', 'D 여유': '#a9e34b'}

        for _, r in sub.iterrows():
            c = clr.get(r['우선순위'], '#aaa')
            fig.add_trace(go.Bar(
                name=r['name'], x=[r['name']],
                y=[r['allocated']], marker_color=c,
                text=f"할당 {r['allocated']:,}", textposition='inside'
            ))
            if r['shortage'] > 0:
                fig.add_trace(go.Bar(
                    name=f"{r['name']} 부족", x=[r['name']],
                    y=[r['shortage']], marker_color='#444',
                    text=f"부족 {r['shortage']:,}", textposition='inside',
                    showlegend=False
                ))

        fig.update_layout(
            barmode='stack',
            title=f'{code} — 재고/재공 할당 상세  |  CE재고:{row["포장재고_CE"]:,}  FDA재고:{row["포장재고_FDA"]:,}  재공:{row["총_재공"]:,}',
            template='plotly_dark', height=380, showlegend=False,
            xaxis_tickangle=30, margin=dict(l=40,r=20,t=70,b=100)
        )
        show_fig(fig, "fig10.html")

        tbl = sub[['우선순위','name','type','ordered','allocated','shortage','fulfill_pct']].copy()
        tbl.columns = ['우선순위','주문명','구분','주문량','할당량','부족량','충족률(%)']
        display(tbl.style
                .format({'주문량':'{:,}','할당량':'{:,}','부족량':'{:,}','충족률(%)':'{:.1f}%'})
                .map(lambda v: 'color:#ff6b6b' if isinstance(v,(int,float)) and v < 0 else '',
                          subset=['부족량']))

alloc_dropdown.observe(show_alloc, names='value')
display(alloc_dropdown, alloc_out)
show_alloc(None)


Dropdown(description='품목 선택:', layout=Layout(width='350px'), options=('C1M3008C', 'C1M3010C', 'C1M3011C', 'C1M…

Output()

---
## 10. 6월 생산계획 (June 1 – 30)
> **Mini** (C1R35\*, C1R30\*): 1 lot = 300 pcs / day  
> **Regular**: 3 lots = 1,200 pcs / day  
> 매일 **잔여 부족수량** 최대 품목부터 배분

In [19]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── 품목 구분 ───────────────────────────────────────────────
def is_mini(code):
    s = str(code)
    return s.startswith('C1M35') or s.startswith('C1M30')

MINI_LOT    = 300
REGULAR_LOT = 400

# ─── 초기 부족수량 (할당 결과 기준) ────────────────────────────
shortage_init = (
    df_alloc.groupby('품목코드')['shortage']
    .sum()
    .reset_index()
    .rename(columns={'shortage': '부족'})
)
shortage_init['mini'] = shortage_init['품목코드'].apply(is_mini)

mini_codes    = set(shortage_init[shortage_init['mini']]['품목코드'])
regular_codes = set(shortage_init[~shortage_init['mini']]['품목코드'])

# ─── 30일 시뮬레이션 ────────────────────────────────────────
START = date(2026, 6, 1)
shortage_cur = shortage_init.set_index('품목코드')['부족'].to_dict()

plan_rows = []
for d in range(30):
    day = START + timedelta(days=d)

    # mini 부족 품목 중 최대
    mini_avail = {k: shortage_cur[k] for k in mini_codes if shortage_cur.get(k, 0) > 0}

    if mini_avail:
        # mini 1 lot + regular 3 lots
        m_item = max(mini_avail, key=mini_avail.get)
        m_qty  = MINI_LOT
        shortage_cur[m_item] = max(0, shortage_cur[m_item] - m_qty)
        reg_lots = 3
    else:
        # mini 부족 없음 → regular 4 lots
        m_item, m_qty = None, 0
        reg_lots = 4

    # regular: 부족 최대 품목부터 1 lot씩
    r_planned = []
    for _ in range(reg_lots):
        reg_avail = {k: shortage_cur.get(k, 0) for k in regular_codes if shortage_cur.get(k, 0) > 0}
        if not reg_avail:
            break
        r_item = max(reg_avail, key=reg_avail.get)
        shortage_cur[r_item] = max(0, shortage_cur[r_item] - REGULAR_LOT)
        r_planned.append((r_item, REGULAR_LOT))

    def slot(lst, i):
        return (lst[i][0], lst[i][1]) if i < len(lst) else (None, 0)

    r1, r2, r3, r4 = slot(r_planned,0), slot(r_planned,1), slot(r_planned,2), slot(r_planned,3)

    plan_rows.append({
        '날짜':          day,
        'Mini_품목':     m_item,
        'Mini_수량':     m_qty,
        'Regular_계획':  r_planned,
        'Reg_품목1': r1[0], 'Reg_수량1': r1[1],
        'Reg_품목2': r2[0], 'Reg_수량2': r2[1],
        'Reg_품목3': r3[0], 'Reg_수량3': r3[1],
        'Reg_품목4': r4[0], 'Reg_수량4': r4[1],
        '잔여_총부족':   sum(shortage_cur.values()),
    })

df_plan = pd.DataFrame(plan_rows)

# ─── 일별 생산계획 테이블 ────────────────────────────────────
disp_cols = ['날짜','Mini_품목','Mini_수량',
             'Reg_품목1','Reg_수량1','Reg_품목2','Reg_수량2',
             'Reg_품목3','Reg_수량3','Reg_품목4','Reg_수량4','잔여_총부족']
df_show = df_plan[disp_cols].copy()
df_show['날짜'] = df_show['날짜'].apply(lambda d: d.strftime('%m/%d (%a)'))
df_show = df_show.fillna('-')

def style_mini(v):
    return 'color:#69db7c; font-weight:bold' if isinstance(v, str) and is_mini(v) else ''

def style_reg(v):
    return 'color:#64b5f6;' if isinstance(v, str) and v != '-' else ''

def style_shortage(v):
    try:
        f = float(v)
        if f == 0:    return 'color:#69db7c; font-weight:bold'
        if f < 5000:  return 'color:#ffa94d;'
        return 'color:#ff6b6b;'
    except:
        return ''

styled = (
    df_show.style
    .set_caption('6월 일별 생산계획  (Mini: C1M35*/C1M30* = 300pcs/lot  |  Regular: 400pcs/lot)')
    .set_table_styles([
        {'selector': 'caption',
         'props': 'font-size:15px; font-weight:bold; padding:10px; text-align:left;'},
        {'selector': 'th',
         'props': 'background:#1e2130; color:#cdd9e5; font-size:11px; padding:6px 10px; text-align:center;'},
        {'selector': 'td',
         'props': 'font-size:11px; padding:5px 10px; text-align:center; border-bottom:1px solid #2d3250;'},
        {'selector': 'tr:nth-child(odd)',  'props': 'background:#1e3a5f;'},
        {'selector': 'tr:nth-child(even)', 'props': 'background:#161b2e;'},
    ])
    .map(style_mini,     subset=['Mini_품목'])
    .map(style_reg,      subset=['Reg_품목1','Reg_품목2','Reg_품목3','Reg_품목4'])
    .map(style_shortage, subset=['잔여_총부족'])
)
display(styled)

,날짜,Mini_품목,Mini_수량,Reg_품목1,Reg_수량1,Reg_품목2,Reg_수량2,Reg_품목3,Reg_수량3,Reg_품목4,Reg_수량4,잔여_총부족
0,06/01 (Mon),C1M3511C,300,C1R4010C,400,C1R4010C,400,C1R4510C,400,-,0,42436
1,06/02 (Tue),C1M3511C,300,C1R4010C,400,C1R4510C,400,C1R4010C,400,-,0,40936
2,06/03 (Wed),C1M3511C,300,C1R4510C,400,C1R4010C,400,C1R4510C,400,-,0,39436
3,06/04 (Thu),C1M3511C,300,C1R4010C,400,C1R4510C,400,C1R4010C,400,-,0,37936
4,06/05 (Fri),C1M3511C,300,C1R4011C,400,C1R4510C,400,C1R4010C,400,-,0,36436
5,06/06 (Sat),C1M3508C,300,C1R4011C,400,C1R4511C,400,C1R4510C,400,-,0,34936
6,06/07 (Sun),C1M3011C,300,C1R4010C,400,C1R4011C,400,C1R4511C,400,-,0,33436
7,06/08 (Mon),C1M3511C,300,C1R4510C,400,C1R4010C,400,C1R4011C,400,-,0,31936
8,06/09 (Tue),C1M3008C,300,C1R4511C,400,C1R4510C,400,C1R4010C,400,-,0,30436
9,06/10 (Wed),C1M3013C,300,C1R4011C,400,C1R4511C,400,C1R4510C,400,-,0,28936


In [20]:
from collections import defaultdict

item_prod = defaultdict(int)
for _, row in df_plan.iterrows():
    if row['Mini_품목']:
        item_prod[row['Mini_품목']] += row['Mini_수량']
    for item, qty in row['Regular_계획']:
        item_prod[item] += qty

item_prod_s = pd.Series(item_prod).sort_values(ascending=False)
colors = ['#69db7c' if is_mini(c) else '#64b5f6' for c in item_prod_s.index]

fig1 = go.Figure(go.Bar(
    x=item_prod_s.index, y=item_prod_s.values,
    marker_color=colors,
    text=[f'{v:,}' for v in item_prod_s.values],
    textposition='outside',
))
fig1.update_layout(
    title='6월 품목별 총 계획생산량',
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117', font_color='#cdd9e5',
    xaxis=dict(title='품목코드', tickangle=-45),
    yaxis=dict(title='계획수량 (pcs)', gridcolor='#2d3250'),
    height=420,
    annotations=[dict(x=0.01, y=1.07, xref='paper', yref='paper',
                      text='<span style="color:#69db7c">■ Mini (C1M35*/C1M30*)</span>  '
                           '<span style="color:#64b5f6">■ Regular</span>',
                      showarrow=False, font_size=12)]
)
show_fig(fig1, "fig11.html")

fig2 = go.Figure(go.Scatter(
    x=df_plan['날짜'].apply(lambda d: d.strftime('%m/%d')),
    y=df_plan['잔여_총부족'],
    mode='lines+markers',
    line=dict(color='#ff6b6b', width=2),
    marker=dict(size=5),
    fill='tozeroy', fillcolor='rgba(255,107,107,0.15)',
    name='잔여 부족',
))
fig2.update_layout(
    title='잔여 총 부족수량 추이 (6/1 → 6/30)',
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117', font_color='#cdd9e5',
    xaxis=dict(title='날짜', tickangle=-45, gridcolor='#2d3250'),
    yaxis=dict(title='잔여 부족수량 (pcs)', gridcolor='#2d3250'),
    height=380,
)
show_fig(fig2, "fig12.html")

init_total  = int(shortage_init['부족'].sum())
final_total = int(df_plan['잔여_총부족'].iloc[-1])
covered     = init_total - final_total
mini_total  = sum(v for k, v in item_prod.items() if is_mini(k))
reg_total   = sum(v for k, v in item_prod.items() if not is_mini(k))

print(f"{'초기 총 부족':15s}: {init_total:>10,} pcs")
print(f"{'6월 계획 커버':15s}: {covered:>10,} pcs  ({covered/init_total*100:.1f}%)")
print(f"{'월말 잔여 부족':15s}: {final_total:>10,} pcs")
print(f"{'Mini 총 생산':15s}: {mini_total:>10,} pcs  ({mini_total//300} lots)")
print(f"{'Regular 총 생산':15s}: {reg_total:>10,} pcs  ({reg_total//400} lots)")

초기 총 부족        :     43,936 pcs
6월 계획 커버       :     43,496 pcs  (99.0%)
월말 잔여 부족       :        440 pcs
Mini 총 생산      :      5,100 pcs  (17 lots)
Regular 총 생산   :     41,200 pcs  (103 lots)


### 10-4. 주문별 부족 품목 수량

In [21]:
# ── 10-4. 주문별 부족 품목 수량 ──────────────────────────────────────────
import plotly.graph_objects as go

# 부족 있는 주문만 필터
df_short = df_alloc[df_alloc['shortage'] > 0].copy()

# ① 주문별 요약 테이블
order_sum = (
    df_short.groupby(['우선순위', 'name', 'type'])
    .agg(부족품목수=('품목코드', 'nunique'),
         총주문=('ordered', 'sum'),
         총부족=('shortage', 'sum'))
    .reset_index()
    .sort_values(['우선순위', 'type'])
)
order_sum['충족률(%)'] = (
    (1 - order_sum['총부족'] / order_sum['총주문']) * 100
).round(1)

def _row_style(row):
    pct = row['충족률(%)']
    if pct < 50:   bg = 'background:#3d0000; color:#ff6b6b'
    elif pct < 80: bg = 'background:#3d2000; color:#ffa94d'
    else:           bg = 'background:#1a3320; color:#69db7c'
    return [bg] * len(row)

styled_sum = (
    order_sum.style
    .set_caption('주문별 부족 현황 요약')
    .format({'총주문': '{:,}', '총부족': '{:,}', '충족률(%)': '{:.1f}%'})
    .apply(_row_style, axis=1)
    .set_table_styles([
        {'selector': 'caption',
         'props': 'font-size:15px;font-weight:bold;padding:8px;text-align:left;'},
        {'selector': 'th',
         'props': 'background:#1e2130;color:#cdd9e5;font-size:11px;padding:6px 12px;text-align:center;'},
        {'selector': 'td',
         'props': 'font-size:11px;padding:5px 12px;text-align:center;border-bottom:1px solid #2d3250;'},
    ])
)
display(styled_sum)

# ② 주문 × 품목 상세 히트맵
pivot = df_alloc.pivot_table(
    index='name', columns='품목코드', values='shortage', aggfunc='sum', fill_value=0
)
# 부족 있는 컬럼만
pivot = pivot.loc[:, (pivot > 0).any()]
# 행 순서 = 우선순위 순
order_priority = df_alloc[['name','우선순위']].drop_duplicates().set_index('name')['우선순위']
sorted_idx = sorted(pivot.index, key=lambda x: order_priority.get(x, 'Z'))
pivot = pivot.loc[sorted_idx]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=list(pivot.columns),
    y=list(pivot.index),
    colorscale=[
        [0,   '#1e2130'],
        [0.01,'#3d2000'],
        [0.3, '#ff6b6b'],
        [1,   '#ff0000'],
    ],
    text=[[f'{v:,}' if v > 0 else '' for v in row] for row in pivot.values],
    texttemplate='%{text}',
    textfont=dict(size=9),
    colorbar=dict(title='부족수량', thickness=12),
    hovertemplate='주문: %{y}<br>품목: %{x}<br>부족: %{z:,}<extra></extra>',
))
fig.update_layout(
    title='주문 × 품목 부족수량 히트맵',
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font_color='#cdd9e5',
    xaxis=dict(title='품목코드', tickangle=-45, tickfont_size=9),
    yaxis=dict(title='주문', autorange='reversed'),
    height=max(350, len(pivot) * 38 + 120),
    margin=dict(l=140, r=20, t=60, b=120),
)
show_fig(fig, 'fig_shortage_heatmap.html', height=max(380, len(pivot)*38+140))

# ③ 주문별 부족 수량 바 차트 (품목 누적)
top_orders = order_sum.nlargest(10, '총부족')
clr_map = {'A 긴급':'#ff6b6b','B 보통':'#ffa94d','C 여유':'#69db7c','D 여유':'#a9e34b'}

fig2 = go.Figure()
for _, r in top_orders.iterrows():
    items = df_short[df_short['name'] == r['name']]
    for _, item_row in items.iterrows():
        fig2.add_trace(go.Bar(
            name=item_row['품목코드'],
            x=[r['name']],
            y=[item_row['shortage']],
            marker_color=clr_map.get(r['우선순위'], '#aaa'),
            hovertemplate=f"{item_row['품목코드']}: {item_row['shortage']:,}pcs<extra></extra>",
            showlegend=False,
        ))
fig2.update_layout(
    barmode='stack',
    title='주문별 품목 부족수량 (상위 10개 주문)',
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font_color='#cdd9e5',
    xaxis=dict(tickangle=-30),
    yaxis=dict(title='부족수량 (pcs)', gridcolor='#2d3250'),
    height=420,
)
show_fig(fig2, 'fig_shortage_bar.html', height=450)

,우선순위,name,type,부족품목수,총주문,총부족,충족률(%)
0,A 긴급,CE_베트남,CE,1,300,63,79.0%
1,B 보통,FDA 이란 ①,FDA,9,"4,860","3,555",26.9%
2,C 여유,CE_두바이,CE,1,80,80,0.0%
3,C 여유,CE_몰도바,CE,2,40,40,0.0%
4,C 여유,FDA 이란 ②,FDA,17,"14,904","9,718",34.8%
5,D 여유,CE_아자르바이젠,CE,2,75,75,0.0%
6,D 여유,FDA 이란 ③,FDA,23,"31,858","30,405",4.6%
